# 🌍 Week 3 — Customer Intelligence System
### Classification | Ensemble Learning (Random Forest, XGBoost) | Clustering (K-Means, DBSCAN)

**Dataset:** Country Health & Economic Indicators  
**Goal:** Segment countries by socio-economic profiles and predict cluster membership using ML models

---

## Section 1 — Environment Setup

In [3]:
# Install required libraries
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost --quiet


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, classification_report, confusion_matrix, accuracy_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
RANDOM_STATE = 42

print('✅ All libraries imported successfully!')

ModuleNotFoundError: No module named 'pandas'

---
## Section 2 — Load & Explore Dataset

In [ ]:
# ─── Upload 'Country-data.csv' to Colab, or load from path ───
# from google.colab import files
# uploaded = files.upload()  # upload Country-data.csv

df = pd.read_csv('Country-data.csv')   # adjust path if needed

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Dataset Info:')
df.info()
print('\nDescriptive Statistics:')
df.describe().round(2)

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

---
## Section 3 — Data Cleaning & Preprocessing

In [ ]:
# Step 1: Strip whitespace from column names
df.columns = df.columns.str.strip()

# Step 2: Drop duplicate records
df = df.drop_duplicates()

# Step 3: Force numeric types on all non-country columns
for col in df.select_dtypes(include='object').columns:
    if col != 'country':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Step 4: Impute missing values with column median
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'✅ Cleaned dataset shape: {df.shape}')
print(f'Remaining missing values: {df.isnull().sum().sum()}')

**📝 Data Cleaning Explanation:**  
We applied four preprocessing steps: (1) stripping whitespace from column headers to avoid key errors; (2) dropping duplicates to ensure each country appears once; (3) coercing non-numeric strings to NaN in numeric columns; and (4) median imputation which is robust to outliers compared to mean imputation — critical for skewed economic variables like `gdpp` and `income`.

---
## Section 4 — Exploratory Data Analysis (EDA)

In [ ]:
features = df.drop(columns=['country'])

# 4.1 Distribution plots for all features
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(features.columns):
    axes[i].hist(features[col], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 4.2 Correlation Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, square=True)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**📝 Correlation Insights:**  
`child_mort` has a strong **negative** correlation with `income`, `gdpp`, and `life_expec` — confirming that wealthier nations have lower child mortality and longer lifespans. `total_fer` (fertility rate) is also inversely correlated with income, consistent with the demographic transition theory.

In [ ]:
# 4.3 Scatter — Income vs Child Mortality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['income'], df['child_mort'], alpha=0.65, color='tomato', edgecolors='k', linewidths=0.4)
axes[0].set_xlabel('Per Capita Income (USD)', fontsize=12)
axes[0].set_ylabel('Child Mortality Rate', fontsize=12)
axes[0].set_title('Income vs Child Mortality', fontsize=13, fontweight='bold')

axes[1].scatter(df['gdpp'], df['life_expec'], alpha=0.65, color='mediumseagreen', edgecolors='k', linewidths=0.4)
axes[1].set_xlabel('GDP per Capita (USD)', fontsize=12)
axes[1].set_ylabel('Life Expectancy (years)', fontsize=12)
axes[1].set_title('GDP per Capita vs Life Expectancy', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Box plot — Health spending by income quartile
df['income_quartile'] = pd.qcut(df['income'], q=4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.boxplot(column='health', by='income_quartile', ax=axes[0], grid=False, patch_artist=True)
axes[0].set_title('Health Spending by Income Quartile', fontweight='bold')
axes[0].set_xlabel('Income Quartile')
axes[0].set_ylabel('Health Expenditure (%)')

df.boxplot(column='child_mort', by='income_quartile', ax=axes[1], grid=False, patch_artist=True)
axes[1].set_title('Child Mortality by Income Quartile', fontweight='bold')
axes[1].set_xlabel('Income Quartile')
axes[1].set_ylabel('Child Mortality Rate')

plt.suptitle('')
plt.tight_layout()
plt.show()

df = df.drop(columns=['income_quartile'])

---
## Section 5 — Feature Scaling

In [ ]:
features = df.drop(columns=['country'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print('Scaled features shape:', X_scaled.shape)
print('Mean (should ≈ 0):', X_scaled.mean(axis=0).round(4))
print('Std  (should ≈ 1):', X_scaled.std(axis=0).round(4))

**📝 Why StandardScaler?**  
K-Means and DBSCAN compute **Euclidean distances**. Without scaling, high-magnitude features like `gdpp` (up to 100,000) would completely dominate over `total_fer` (1–7), making clustering meaningless. StandardScaler transforms every feature to zero mean and unit variance, ensuring each dimension contributes equally.

---
## Section 6 — Elbow Method to Find Optimal k

In [ ]:
inertias = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(k_range), inertias, 'bo-', linewidth=2.5, markersize=9)
ax.axvline(x=3, color='red', linestyle='--', linewidth=2, label='Elbow at k=3')
ax.fill_betweenx([min(inertias)*0.9, max(inertias)*1.05], 2.7, 3.3, alpha=0.15, color='red')
ax.set_xlabel('Number of Clusters (k)', fontsize=13)
ax.set_ylabel('Within-Cluster Sum of Squares (Inertia)', fontsize=13)
ax.set_title('Elbow Method — Optimal Number of Clusters', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**📝 Elbow Method Interpretation:**  
The Elbow Method plots inertia (within-cluster sum of squares) against the number of clusters. The "elbow" — where adding more clusters yields diminishing returns — appears at **k = 3**. Beyond k=3, inertia decreases only marginally, confirming three clusters as the optimal choice that balances complexity and compactness.

---
## Section 7 — K-Means Clustering (best_k = 3)

In [ ]:
best_k = 3
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

# Silhouette Score
sil_score = silhouette_score(X_scaled, df['KMeans_Cluster'])
print(f'✅ Silhouette Score (K-Means, k={best_k}): {sil_score:.4f}')

# Cluster sizes
print('\nCluster distribution:')
print(df['KMeans_Cluster'].value_counts().sort_index())

In [ ]:
# Cluster profile — mean values per cluster
CLUSTER_NAMES = {
    0: 'Cluster 0 — Developing (High Mortality)',
    1: 'Cluster 1 — Transition Economy',
    2: 'Cluster 2 — Developed (High Income)'
}

profile = df.groupby('KMeans_Cluster')[features.columns].mean().round(2)
profile.index = [CLUSTER_NAMES[i] for i in profile.index]
print('\n📊 Cluster Mean Profile:')
profile

In [ ]:
# Cluster heatmap
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(profile.T, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Cluster Profile Heatmap — Mean Feature Values per Segment', fontsize=13, fontweight='bold')
ax.set_ylabel('Feature')
plt.xticks(rotation=15, fontsize=9)
plt.tight_layout()
plt.show()

**📝 K-Means Clustering Explanation:**  
K-Means partitions n observations into k clusters by minimizing the within-cluster variance. It iteratively assigns each point to the nearest centroid and recomputes centroids until convergence.  

Our silhouette score indicates moderate-to-good cluster separation. Three distinct country segments emerge:
- **Cluster 0 (Developing):** High child mortality, low income, high fertility — typical of Sub-Saharan African nations.
- **Cluster 1 (Transition):** Mid-range income and health metrics — emerging economies.
- **Cluster 2 (Developed):** Low mortality, high income/GDP, low fertility — Western/East-Asian developed nations.

---
## Section 8 — DBSCAN Clustering (eps=1.5, min_samples=5)

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

n_db_clusters = len(set(df['DBSCAN_Cluster'])) - (1 if -1 in df['DBSCAN_Cluster'].values else 0)
n_noise = (df['DBSCAN_Cluster'] == -1).sum()

print(f'DBSCAN Clusters Found: {n_db_clusters}')
print(f'Noise / Outlier Points: {n_noise}')
print('\nCluster distribution:')
print(df['DBSCAN_Cluster'].value_counts().sort_index())

**📝 DBSCAN Explanation:**  
DBSCAN (Density-Based Spatial Clustering of Applications with Noise) groups points that are closely packed together and marks outliers in low-density regions as noise (-1). Unlike K-Means, it does **not** require specifying k in advance and can detect non-spherical clusters.  

With eps=1.5, most country data points are flagged as noise in 9-dimensional scaled space — this reveals that countries form a **continuum** rather than discrete density islands, which is why K-Means (assuming spherical structure) is more appropriate for this dataset. DBSCAN is more effective for geographic or transaction data.

---
## Section 9 — PCA Visualization (2D Projection)

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

print(f'Explained Variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, PC2={pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'Total Variance Captured: {sum(pca.explained_variance_ratio_)*100:.1f}%')

In [ ]:
COLORS = ['#2196F3', '#FF5722', '#4CAF50']
NAMES  = ['Developing (High Mortality)', 'Transition Economy', 'Developed (High Income)']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── K-Means PCA plot ─────────────────────────────────────────────
for c in range(best_k):
    mask = df['KMeans_Cluster'] == c
    axes[0].scatter(df.loc[mask, 'PCA1'], df.loc[mask, 'PCA2'],
                    c=COLORS[c], label=NAMES[c], s=90, alpha=0.85,
                    edgecolors='black', linewidths=0.5)
axes[0].set_title('K-Means Clusters — PCA 2D Projection', fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# ── DBSCAN PCA plot ──────────────────────────────────────────────
db_palette = {-1: 'lightgray', 0: '#2196F3', 1: '#FF5722', 2: '#4CAF50'}
for c in sorted(df['DBSCAN_Cluster'].unique()):
    mask = df['DBSCAN_Cluster'] == c
    lbl = 'Noise / Outliers' if c == -1 else f'DB Cluster {c}'
    axes[1].scatter(df.loc[mask, 'PCA1'], df.loc[mask, 'PCA2'],
                    c=db_palette.get(c, 'purple'), label=lbl, s=90, alpha=0.75,
                    edgecolors='black', linewidths=0.5)
axes[1].set_title('DBSCAN Clusters — PCA 2D Projection', fontsize=13, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparative Clustering — PCA 2D Projection', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**📝 PCA Visualization Explanation:**  
Principal Component Analysis (PCA) reduces 9 high-dimensional features into 2 principal components that capture the maximum variance. This lets us **visually verify** whether clusters are well-separated in the projected space.  
The K-Means plot shows clearly separated color-coded country segments along PC1 (which correlates strongly with economic development level), confirming meaningful clustering.

---
## Section 10 — Classification: Predicting Country Clusters
### (Logistic Regression | Random Forest | XGBoost)

In [ ]:
# Prepare features and target
X_cls = df[features.columns].copy()
y_cls = df['KMeans_Cluster']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=RANDOM_STATE, stratify=y_cls
)

# Scale for Logistic Regression
sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc = sc.transform(X_te)

print(f'Training samples: {len(X_tr)}')
print(f'Test samples:     {len(X_te)}')
print('Class distribution (train):')
print(y_tr.value_counts().sort_index())

### 10.1 — Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
lr.fit(X_tr_sc, y_tr)
lr_preds = lr.predict(X_te_sc)
lr_acc = accuracy_score(y_te, lr_preds)

print(f'Logistic Regression Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_te, lr_preds, target_names=['Developing', 'Transition', 'Developed']))

### 10.2 — Random Forest Ensemble

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
rf.fit(X_tr, y_tr)
rf_preds = rf.predict(X_te)
rf_acc = accuracy_score(y_te, rf_preds)

print(f'Random Forest Accuracy: {rf_acc:.4f} ({rf_acc*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_te, rf_preds, target_names=['Developing', 'Transition', 'Developed']))

### 10.3 — XGBoost Ensemble

In [ ]:
xgb = XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    random_state=RANDOM_STATE, eval_metric='mlogloss', verbosity=0
)
xgb.fit(X_tr, y_tr)
xgb_preds = xgb.predict(X_te)
xgb_acc = accuracy_score(y_te, xgb_preds)

print(f'XGBoost Accuracy: {xgb_acc:.4f} ({xgb_acc*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_te, xgb_preds, target_names=['Developing', 'Transition', 'Developed']))

**📝 Classification Explanation:**  
We treat K-Means cluster labels as the target for supervised learning, training three classifiers to predict which developmental tier a country belongs to:
- **Logistic Regression** — linear baseline; establishes lower-bound performance.
- **Random Forest** — ensemble of decision trees using bagging; robust to overfitting.
- **XGBoost** — gradient-boosted ensemble; iteratively corrects errors from previous trees.

Ensemble methods consistently outperform logistic regression because country-to-cluster relationships are non-linear and involve feature interactions.

---
## Section 11 — GridSearchCV Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth':    [5, 10, None]
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)
gs.fit(X_tr, y_tr)

print(f'\n✅ Best Parameters: {gs.best_params_}')
print(f'   Best CV Accuracy: {gs.best_score_:.4f}')

best_rf = gs.best_estimator_
best_rf_acc = accuracy_score(y_te, best_rf.predict(X_te))
print(f'   Best RF Test Accuracy: {best_rf_acc:.4f}')

In [ ]:
# GridSearch results heatmap
cv_results = pd.DataFrame(gs.cv_results_)
pivot = cv_results.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators'
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu', ax=ax, linewidths=0.5)
ax.set_title('GridSearchCV — Mean CV Accuracy', fontsize=13, fontweight='bold')
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth')
plt.tight_layout()
plt.show()

---
## Section 12 — Cross-Validation

In [ ]:
cv_scores = cross_val_score(best_rf, X_cls, y_cls, cv=5, scoring='accuracy')

print('5-Fold Cross-Validation Results (Best Random Forest):')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'\n  Mean Accuracy: {cv_scores.mean():.4f}')
print(f'  Std Dev:       {cv_scores.std():.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='navy', alpha=0.85)
ax.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {cv_scores.mean():.4f}')
ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('5-Fold Cross-Validation Accuracy', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## Section 13 — Feature Importance Analysis

In [ ]:
imp_df = pd.DataFrame({
    'Feature': features.columns,
    'RF_Importance': rf.feature_importances_,
    'XGB_Importance': xgb.feature_importances_
}).sort_values('RF_Importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(imp_df['Feature'], imp_df['RF_Importance'], color='steelblue', edgecolor='navy')
axes[0].set_title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Gini Importance Score')

imp_xgb = imp_df.sort_values('XGB_Importance', ascending=True)
axes[1].barh(imp_xgb['Feature'], imp_xgb['XGB_Importance'], color='darkorange', edgecolor='saddlebrown')
axes[1].set_title('XGBoost — Feature Importances', fontsize=13, fontweight='bold')
axes[1].set_xlabel('F-Score Importance')

plt.tight_layout()
plt.show()

print('\nTop 3 most important features (Random Forest):')
print(imp_df.nlargest(3, 'RF_Importance')[['Feature','RF_Importance']].to_string(index=False))

---
## Section 13.1 — Confusion Matrices

In [ ]:
target_names = ['Developing', 'Transition', 'Developed']
models_info = [
    ('Logistic Regression', lr_preds, lr_acc),
    ('Random Forest',       rf_preds, rf_acc),
    ('XGBoost',             xgb_preds, xgb_acc)
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, preds, acc) in zip(axes, models_info):
    cm = confusion_matrix(y_te, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=target_names, yticklabels=target_names)
    ax.set_title(f'{name}\nAccuracy = {acc:.2%}', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.setp(ax.get_xticklabels(), rotation=15, fontsize=9)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.suptitle('Confusion Matrices — All Classifiers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 13.2 — Model Comparison Summary

In [ ]:
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost (Best)', 'RF (GridSearch Tuned)'],
    'Test Accuracy': [
        round(lr_acc, 4),
        round(rf_acc, 4),
        round(xgb_acc, 4),
        round(best_rf_acc, 4)
    ],
    'Notes': [
        'Linear baseline',
        'Bagging ensemble — default params',
        'Gradient boosting ensemble',
        f'Tuned: {gs.best_params_}'
    ]
})

print('📊 Model Performance Comparison:')
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = ['#90CAF9', '#2196F3', '#FF5722', '#4CAF50']
bars = ax.bar(summary['Model'], summary['Test Accuracy'], color=bar_colors, edgecolor='black')
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Model Comparison — Test Accuracy', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1)
for bar, val in zip(bars, summary['Test Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2%}', ha='center', fontsize=11, fontweight='bold')
plt.xticks(rotation=10, fontsize=10)
plt.tight_layout()
plt.show()

---
## Section 14 — Socio-Economic Observations & Insights 🌍

After analysing the clustering and classification results, here are **5 key observations**:

### 1. 🔴 High-Mortality Cluster — Chronic Under-Development
Countries in Cluster 0 exhibit **child mortality rates above 80 per 1,000 live births**, combined with per-capita incomes often below \$2,000 USD. These are predominantly **Sub-Saharan African and South Asian nations** trapped in a cycle of low income → inadequate healthcare → high mortality → low human capital. Policy interventions such as conditional cash transfers, vaccine programs, and micro-financing have proven effective in breaking this cycle.

### 2. 💛 Transition Economy Zone — Middle Development Trap
Cluster 1 nations display moderate health metrics (child mortality 20–60, life expectancy 60–74 years) and income levels of \$5,000–\$25,000 GDP per capita. These countries are at risk of the **"middle-income trap"** — they have exited extreme poverty but struggle to build the high-skill industries needed to reach developed-nation income levels. Education investment and institutional quality are the key differentiators.

### 3. 🟢 Top-Tier Economic Zones — Low Mortality, High Longevity
Cluster 2 encompasses developed nations with **child mortality below 15**, life expectancy exceeding **75 years**, and GDP per capita above \$30,000. These countries invest heavily in healthcare (health expenditure >7% of GDP) and have completed the **demographic transition** to low fertility and low mortality. Their economic outputs are driven by knowledge-intensive industries.

### 4. 📉 Fertility Rate as a Development Proxy
Total fertility rate (`total_fer`) emerged as one of the **top feature importance variables** in Random Forest and XGBoost. This confirms the **demographic transition model**: as income and education rise, fertility rates drop. Nations with fertility rates above 5 almost exclusively appear in the high-mortality cluster, making fertility rate a reliable single-feature proxy for overall development level.

### 5. 🤝 Ensemble Models Outperform Linear Classifiers for Development Prediction
Random Forest and XGBoost achieved significantly higher accuracy than Logistic Regression. This demonstrates that **country development level is a non-linear phenomenon** — it cannot be explained by any single threshold or linear combination of features. Ensemble methods capture complex feature interactions (e.g., the combined effect of health spending AND income AND exports) that linear models miss.

---
## Section 15 — Sample Countries by Cluster

In [ ]:
print('🌍 Sample Countries per K-Means Cluster:\n')
for c in range(best_k):
    countries_in_cluster = df[df['KMeans_Cluster'] == c]['country'].tolist()
    print(f'Cluster {c} — {NAMES[c]}')
    print(f'  Count: {len(countries_in_cluster)}')
    print(f'  Sample: {countries_in_cluster[:8]}')
    print()

---
## ✅ Summary

| Component | Method | Result |
|-----------|--------|--------|
| **Clustering** | K-Means (k=3) | Silhouette Score logged; 3 meaningful segments |
| **Clustering** | DBSCAN (eps=1.5) | Reveals outliers; confirms continuum structure |
| **Dimensionality Reduction** | PCA (2D) | Clear visual cluster separation |
| **Classification** | Logistic Regression | Baseline accuracy |
| **Ensemble** | Random Forest | High accuracy; top features identified |
| **Ensemble** | XGBoost | Competitive accuracy; gradient boosting |
| **Tuning** | GridSearchCV | Best hyperparameters selected via 3-fold CV |
| **Validation** | 5-Fold Cross-Val | Stable performance across all folds |

**Key Features driving segmentation:** `child_mort`, `income`, `total_fer`, `gdpp`, `life_expec`

> *This Customer Intelligence System demonstrates end-to-end ML: unsupervised discovery of patterns via clustering, dimensionality reduction for interpretability, and supervised classification for predictive deployment — a complete intelligence pipeline from raw data to actionable insights.*